## Step 3.3 — Unit normalization & material-level CO₂ calculation


### Results Snapshot (2022)

- Included materials: Glass, Aluminium, Plastic (modeled as PET proxy)
- Estimated avoided emissions (total): **~0.99 million tonnes CO₂e**
- Method: avoided-burden screening estimate using conservative EU-average factors


##### Interpretation — Environmental Impact (E in ESG)

Using a conservative avoided-burden approach, the German Pfand system’s recycling of selected beverage packaging materials in 2022 is estimated to have avoided approximately 1.0 million tonnes of CO₂e emissions.
The majority of avoided emissions originate from recycled plastics (modeled using PET as a proxy), reflecting both high material throughput and substantial differences between virgin and recycled production emissions. Aluminium contributes a disproportionately high share of avoided emissions relative to its mass, due to its energy-intensive primary production. Glass shows a smaller but still material contribution.
These estimates represent screening-level environmental benefits, based on EU-average life-cycle factors and conservative assumptions, and should be interpreted as indicative rather than as full product-level life-cycle footprints.

##### Scope & limitations

Avoided emissions are estimated using average EU life-cycle factors rather than Germany-specific production pathways.
Plastic recycling is modeled using PET as a proxy for the Pfand plastic stream; actual polymer composition may vary.
Collection, sorting, and logistics emissions are not explicitly modeled.
Results reflect material-level avoided emissions, not system-wide net carbon footprints.

In [1]:
import pandas as pd
#Master dataset
master = pd.read_csv("../data/cleaned/pfand_2022_master.csv")
#ESG constants
constants = pd.read_csv("../data/esg/esg_constants.csv")
master, constants

eligible_materials = ["Glas", "Aluminium", "Kunststoff"]
esg_df = master[master["material"].isin(eligible_materials)].copy()
esg_df

material_map = {
    "Glas": "GLASS",
    "Aluminium": "ALU",
    "Kunststoff": "PET"  # proxy assumption, already documented
}
esg_df["material_key"] = esg_df["material"].map(material_map)
esg_df

esg_df["recycled_kg"] = esg_df["total_recycled_kt"] * 1_000_000
esg_df = esg_df.merge( constants[["material_key", "co2e_saved_kg_per_kg"]], on="material_key", how="left")
esg_df

esg_df["co2e_avoided_kg"] = (esg_df["recycled_kg"] * esg_df["co2e_saved_kg_per_kg"])
esg_df[[
    "material",
    "total_recycled_kt",
    "recycled_kg",
    "co2e_saved_kg_per_kg",
    "co2e_avoided_kg"
]]
esg_df["co2e_avoided_tonnes"] = esg_df["co2e_avoided_kg"] / 1_000
esg_df[[
    "material",
    "total_recycled_kt",
    "co2e_avoided_tonnes"
]]
total_co2e_avoided_tonnes = esg_df["co2e_avoided_tonnes"].sum()
total_co2e_avoided_tonnes
summary = esg_df[[
    "material",
    "total_recycled_kt",
    "co2e_avoided_tonnes"
]].copy()
summary["share_pct"] = ( summary["co2e_avoided_tonnes"] / summary["co2e_avoided_tonnes"].sum() * 100 )
summary

# Keeping final columns
phase3_out = esg_df[[
    "material",
    "total_recycled_kt",
    "co2e_saved_kg_per_kg",
    "co2e_avoided_tonnes"
]].copy()
phase3_out.to_csv("../data/cleaned/phase3_co2e_avoided_2022_by_material.csv", index=False)
phase3_out_display = phase3_out.copy()
phase3_out_display.index = range(1, len(phase3_out_display) + 1)
phase3_out_display



,material,total_recycled_kt,co2e_saved_kg_per_kg,co2e_avoided_tonnes
1,Glas,35.1,0.67,23517.0
2,Aluminium,50.0,6.10,305000.0
3,Kunststoff,441.7,1.50,662550.0


## Step 3.5 — Waste diversion by material


In [2]:
# Waste diversion is equivalent to recycled quantity for Pfand materials
waste_diversion = esg_df[[
    "material",
    "total_recycled_kt"
]].copy()

waste_diversion = waste_diversion.rename(
    columns={"total_recycled_kt": "waste_diverted_kt"}
)
waste_diversion.index = range(1, len(waste_diversion) + 1)
waste_diversion


,material,waste_diverted_kt
1,Glas,35.1
2,Aluminium,50.0
3,Kunststoff,441.7


## Step 3.6 — Pfand circulation (€0.25 system)

Due to the absence of publicly available national container-count data, Pfand circulation is estimated by converting recycled mass into approximate container counts using average container weights. These estimates provide order-of-magnitude insight into the economic scale of the system rather than exact accounting figures.
### Assumptions — Average container weights
- Glass bottle: 500 g  
- Aluminium can: 15 g  
- Plastic bottle (PET proxy): 25 g  

Pfand value per container: €0.25


In [3]:
# Average container weights in kg (assumptions)
avg_container_weights = {
    "Glas": 0.50,        # 500 g glass bottle
    "Aluminium": 0.015,  # 15 g aluminium can
    "Kunststoff": 0.025 # 25 g PET bottle (proxy)
}

PFAND_VALUE_EUR = 0.25

pfand_circulation = esg_df[["material", "recycled_kg"]].copy()

pfand_circulation["avg_container_weight_kg"] = (
    pfand_circulation["material"].map(avg_container_weights)
)

# Estimate number of containers
pfand_circulation["estimated_containers"] = (
    pfand_circulation["recycled_kg"] / pfand_circulation["avg_container_weight_kg"]
)

# Estimate Pfand circulation value (€)
pfand_circulation["pfand_value_eur"] = (
    pfand_circulation["estimated_containers"] * PFAND_VALUE_EUR
)

pfand_circulation[[
    "material",
    "estimated_containers",
    "pfand_value_eur"
]]


,material,estimated_containers,pfand_value_eur
0,Glas,7.020000e+07,1.755000e+07
1,Aluminium,3.333333e+09,8.333333e+08
2,Kunststoff,1.766800e+10,4.417000e+09


## Step 3.7 — Unclaimed Pfand estimation

Unclaimed Pfand is estimated based on material-specific return rates. 
The unclaimed share represents deposits paid by consumers for containers 
that were not returned into the Pfand system.


In [4]:
# Bring recycling / return rates
return_rates = master[[
    "material",
    "recycling_rate_pct"
]].copy()

return_rates["recycling_rate_pct"] = (
    return_rates["recycling_rate_pct"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)

return_rates["recycling_rate_pct"] = pd.to_numeric(
    return_rates["recycling_rate_pct"],
    errors="coerce"
)

return_rates

# Merge with Pfand circulation results
unclaimed_pfand = pfand_circulation.merge(
    return_rates,
    on="material",
    how="left"
)

unclaimed_pfand[["material", "recycling_rate_pct"]]

# Convert % to fraction
unclaimed_pfand["return_rate"] = unclaimed_pfand["recycling_rate_pct"] / 100

# Estimate unclaimed Pfand (€)
unclaimed_pfand["unclaimed_pfand_eur"] = (
    unclaimed_pfand["pfand_value_eur"] * (1 - unclaimed_pfand["return_rate"])
)

unclaimed_pfand[[
    "material",
    "pfand_value_eur",
    "recycling_rate_pct",
    "unclaimed_pfand_eur"
]]




,material,pfand_value_eur,recycling_rate_pct,unclaimed_pfand_eur
0,Glas,1.755000e+07,99.2,1.404000e+05
1,Aluminium,8.333333e+08,99.2,6.666667e+06
2,Kunststoff,4.417000e+09,95.3,2.075990e+08


In [5]:
economic_flow = unclaimed_pfand[[
    "material",
    "pfand_value_eur",
    "recycling_rate_pct",
    "unclaimed_pfand_eur"
]].copy()

economic_flow.to_csv(
    "../data/cleaned/economic_flow_results_2022.csv",
    index=False
)

economic_flow


,material,pfand_value_eur,recycling_rate_pct,unclaimed_pfand_eur
0,Glas,1.755000e+07,99.2,1.404000e+05
1,Aluminium,8.333333e+08,99.2,6.666667e+06
2,Kunststoff,4.417000e+09,95.3,2.075990e+08


In [6]:
co2_by_material = pd.read_csv("../data/cleaned/phase3_co2e_avoided_2022_by_material.csv")

co2_by_material.to_csv("../data/cleaned/dashboard_co2_by_material_2022.csv", index=False)
